# Colab Intro

Notebook zawiera przykładowe komórki wymagane w zadaniu: obliczenia, zapisy plików, wykresy, animacja i krzywa parametryczna.

## Sumowanie ciągu geometrycznego (przykład)

Dla ciągu geometrycznego o pierwszym wyrazie $a$ i ilorazie $r$ suma pierwszych $n$ wyrazów wynosi:
$$S_n = a \frac{1-r^n}{1-r}, \quad r \neq 1.$$

Przykładowy kod w Pythonie (prawidłowo renderowany w Markdown):
```python
def geom_sum(a, r, n):
    if r == 1:
        return a * n
    return a * (1 - r**n) / (1 - r)
```


In [ ]:
# 2 + 2
print('2 + 2 =', 2 + 2)

In [ ]:
# pi ** pi
import math
print('pi ** pi =', math.pi ** math.pi)

In [ ]:
# Suma kwadratów od 1 do 100
s = sum(k*k for k in range(1, 101))
print('Suma kwadratów 1..100 =', s)

In [ ]:
# Zapis wyniku do pliku colab_wynik.txt i pokazanie katalogu roboczego
import os
filename = 'colab_wynik.txt'
with open(filename, 'w', encoding='utf-8') as f:
    f.write(str(s) + '\n')
print('Zapisano', filename, 'w katalogu', os.getcwd())

In [ ]:
# Wykres y = k^2 dla k = 1..20 oraz zapis do wykres.png
import matplotlib.pyplot as plt
k = list(range(1, 21))
y = [i*i for i in k]
plt.figure(figsize=(6,4))
plt.plot(k, y, marker='o')
plt.title('y = k^2 dla k = 1..20')
plt.xlabel('k')
plt.ylabel('y')
plt.grid(True)
plt.savefig('wykres.png', dpi=150)
plt.show()
print('Zapisano wykres do wykres.png')

In [ ]:
# Generowanie animacji wahadła i zapis do video.mp4
# W Colabie może być konieczne zainstalowanie ffmpeg: !apt-get -y install ffmpeg
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, Video

# Parametry prostego wahadła
g = 9.81
L = 1.0
theta0 = 0.9  # początkowe wychylenie (rad)
omega0 = 0.0
t = np.linspace(0, 10, 500)

# Proste rozwiązanie przy małych wychyleniach (harmoniczne)
omega = np.sqrt(g / L)
theta = theta0 * np.cos(omega * t)
x = L * np.sin(theta)
y = -L * np.cos(theta)

fig, ax = plt.subplots(figsize=(5,5))
ax.set_xlim(-L-0.2, L+0.2)
ax.set_ylim(-L-0.2, L+0.2)
line, = ax.plot([], [], lw=2)
point, = ax.plot([], [], 'o', color='red')

def init():
    line.set_data([], [])
    point.set_data([], [])
    return line, point

def animate(i):
    thisx = [0, x[i]]
    thisy = [0, y[i]]
    line.set_data(thisx, thisy)
    point.set_data(x[i], y[i])
    return line, point

anim = animation.FuncAnimation(fig, animate, init_func=init, frames=len(t), interval=20, blit=True)

# Zapis do pliku mp4 - próbujemy bezpiecznie obsłużyć brak ffmpeg
mp4_path = 'video.mp4'
saved_as_gif = False
try:
    # Preferujemy FFMpegWriter jeśli dostępny
    try:
        Writer = animation.writers['ffmpeg']
        writer = Writer(fps=30, metadata=dict(artist='Me'), bitrate=1800)
        anim.save(mp4_path, writer=writer)
        print('Zapisano animację do', mp4_path)
    except Exception:
        # Fallback: spróbuj użyć FFMpegWriter klasy bezpośrednio
        try:
            from matplotlib.animation import FFMpegWriter
            writer = FFMpegWriter(fps=30, metadata=dict(artist='Me'), bitrate=1800)
            anim.save(mp4_path, writer=writer)
            print('Zapisano animację do', mp4_path)
        except Exception as e_ffmpeg:
            # Jeśli ffmpeg nie jest dostępne, zapisujemy bezpośrednio jako GIF (Pillow)
            try:
                from matplotlib.animation import PillowWriter
                gif_path = 'video.gif'
                anim.save(gif_path, writer=PillowWriter(fps=30))
                saved_as_gif = True
                print('ffmpeg nie jest dostępny; zapisano animację jako GIF do', gif_path)
            except Exception as e_pillow:
                print('Nie udało się zapisać animacji (ffmpeg i Pillow niedostępne):', e_ffmpeg, e_pillow)
finally:
    plt.close(fig)

# Wyświetlenie osadzonego wideo (jeśli środowisko wspiera)
if not saved_as_gif and os.path.exists(mp4_path):
    try:
        display(Video(mp4_path, embed=True, width=400))
    except Exception:
        # W niektórych środowiskach Video nie wyświetla się automatycznie
        print('Gotowy plik:', mp4_path)
elif saved_as_gif and os.path.exists('video.gif'):
    try:
        from IPython.display import Image
        display(Image('video.gif'))
    except Exception:
        print('Gotowy plik: video.gif')

In [ ]:
# Konwersja video.mp4 -> video.gif (imageio lub ffmpeg)
import os
import imageio
mp4_path = 'video.mp4'
gif_path = 'video.gif'
if os.path.exists(mp4_path):
    try:
        reader = imageio.get_reader(mp4_path)
        meta = reader.get_meta_data() if hasattr(reader, 'get_meta_data') else {}
        fps = meta.get('fps', 30)
        writer = imageio.get_writer(gif_path, fps=fps)
        for frame in reader:
            writer.append_data(frame)
        writer.close()
        print('Zapisano GIF do', gif_path)
    except Exception as e:
        print('Błąd konwersji do GIF:', e)
else:
    print('Plik', mp4_path, 'nie istnieje — pomijam konwersję do GIF.')

In [ ]:
# Modyfikacja krzywej parametrycznej i zapis do parametric_curve.png
# Zamiast sferycznej krzywej użyjemy krzywej Lissajous w 3D
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

t = np.linspace(0, 2*np.pi, 1000)
a, b, c = 3, 4, 5
x = np.sin(a * t + 0.0)
y = np.sin(b * t + np.pi/4)
z = np.sin(c * t + np.pi/2)

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(x, y, z, lw=1.5, color='purple')
ax.set_title('Parametric Lissajous 3D')
ax.set_xlabel('x(t)')
ax.set_ylabel('y(t)')
ax.set_zlabel('z(t)')
plt.savefig('parametric_curve.png', dpi=150)
plt.show()
print('Zapisano krzywą parametryczną do parametric_curve.png')